In [ ]:
import re
import requests
from bs4 import BeautifulSoup
import json
from sentence_transformers import SentenceTransformer
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM
from typing import List, Dict, Any

In [ ]:
# Загрузка словаря микрокатегорий
def load_mc_mapping(json_path='dic_of_mc_key_phrases.json'):
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        # Создаём словарь {mcTitle: mcId}
        return {item['mcTitle']: item['mcId'] for item in data}
    except FileNotFoundError:
        print(f'Ошибка: файл {json_path} не найден')
        return {}
    except json.JSONDecodeError:
        print('Ошибка: некорректный JSON')
        return {}

# Инициализация сессии
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7',
    'Referer': 'https://www.avito.ru/',
}

session = requests.Session()
session.cookies.set('_ga', 'dummy')
session.cookies.set('_ym_uid', 'dummy')

url = input('Ссылка на объявление: ').strip()

try:
    resp = session.get(url, headers=headers, timeout=10)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')

    # --- Парсинг ID ---
    id_span = soup.find('span', {'data-marker': 'item-view/item-id'})
    if id_span:
        text = id_span.get_text(strip=True)
        match = re.search(r'(\d+)', text)
        item_id = int(match.group(1)) if match else None
    else:
        item_id = None

    # --- Парсинг описания ---
    desc_div = soup.find('div', {'data-marker': 'item-view/item-description'})
    if desc_div:
        description = ' '.join(desc_div.stripped_strings)
    else:
        description = 'Описание не найдено'

    # --- Парсинг микрокатегории (название) ---
    # Ищем все элементы breadcrumb с itemprop="name"
    name_spans = soup.find_all('span', {'itemprop': 'name'})
    mc_title = None
    if name_spans:
        # Последний элемент — текущая категория
        mc_title = name_spans[-1].get_text(strip=True)
    else:
        # Альтернативный поиск через data-marker (если структура иная)
        breadcrumb = soup.find('div', {'data-markers': 'breadcrumbs'})
        if breadcrumb:
            last_link = breadcrumb.find_all('a')[-1] if breadcrumb.find_all('a') else None
            if last_link:
                mc_title = last_link.get_text(strip=True)

    # --- Сопоставление с JSON ---
    mc_mapping = load_mc_mapping()  # загружаем соответствия
    if mc_title and mc_title in mc_mapping:
        mc_id = mc_mapping[mc_title]
    else:
        mc_id = "unknown"
        if not mc_title:
            mc_title = "Категория не найдена"

    # --- Итоговый словарь ---
    classified = {
        "itemId": item_id,
        "mcId": mc_id,
        "mcTitle": mc_title,
        "description": description
    }
    print(classified)

except Exception as e:
    print(f'Ошибка: {e}')

In [ ]:
def clean_description(description):
    # Оставляем только:
    # - буквы: a-z A-Z а-я А-Я ё Ё
    # - цифры: 0-9
    # - пробелы и стандартные знаки препинания
    pattern = r'[^a-zA-Zа-яА-ЯёЁ0-9\s\.,!?;:()\[\]{}\'"-+=/&*#@$%~`_—-]'
    cleaned = re.sub(pattern, '', description)
    # Убираем лишние пробелы (множественные заменяем на один)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

# Применяем
description_raw = classified['description']
cleaned_description = clean_description(description_raw)

print(cleaned_description)

In [ ]:
def normalize_text(text):
    # 1. Приводим к нижнему регистру
    text = text.lower()
    
    # 2. Уменьшаем повторяющиеся буквы (3+ одинаковых подряд → 2)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    
    # Удаляем повторяющиеся знаки препинания
    text = re.sub(r'([.,!?;:"-+=/&*#@$%~`_—-]){2,}', r'\1', text)
    
    return text

# Применяем к cleaned_description
normalized_description = normalize_text(cleaned_description)

print(normalized_description)

In [ ]:
def split_into_chunks(text):
    # делим по всем основным разделителям
    chunks = re.split(r'[.!?;,:\(\)\[\]\{\}/\—\_\#]|\sи\s|\sили\s', text)

    # чистим пробелы и убираем пустые чанки
    clean_chunks = []

    for c in chunks:
        c = c.strip()
        if not c:
            continue

        # удалить только цифры
        if re.fullmatch(r'\d+', c):
            continue

        # удалить одиночные бесполезные слова
        if c in {"я", "мы"}:
            continue

        clean_chunks.append(c)

    return clean_chunks

# Применяем к нормализованному тексту
chunks_of_description = split_into_chunks(normalized_description)

print(chunks_of_description)

In [ ]:
# Загрузка словаря
def load_categories(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)
    
# Подготовка embeddings для категорий
model = SentenceTransformer("deepvk/USER2-base")

def prepare_category_embeddings(categories):
    for cat in categories:
        cat["embeddings"] = model.encode(cat["keyPhrases"])
    return categories

# Функция similarity
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Матчинк одного чанка
def match_chunk(chunk, categories, threshold=0.75):
    chunk_emb = model.encode([chunk])[0]

    best_score = 0
    best_cat = None

    for cat in categories:
        for emb in cat["embeddings"]:
            score = cosine_sim(chunk_emb, emb)

            if score > best_score:
                best_score = score
                best_cat = cat

    if best_score >= threshold:
        return {
            "chunk": chunk,
            "mcId": best_cat["mcId"],
            "mcTitle": best_cat["mcTitle"],
            "score": float(best_score),
            "matched": True
        }

    return {
    "chunk": chunk,
    "mcId": best_cat["mcId"] if best_cat else None,
    "mcTitle": best_cat["mcTitle"] if best_cat else None,
    "score": float(best_score),
    "matched": False
    }

# Обработка всех чанков
def classify_chunks(chunks, categories):
    results = []

    for chunk in chunks:
        match = match_chunk(chunk, categories)
        results.append(match)

    return results

In [ ]:
categories = load_categories("dic_of_mc_key_phrases.json")
categories = prepare_category_embeddings(categories)

chunk_classification = classify_chunks(chunks_of_description, categories)
matched = [c for c in chunk_classification if c["matched"]]
unmatched = [c for c in chunk_classification if not c["matched"]]

print("\n=== MATCHED ===")
for c in matched:
    print(c)

print("\n=== UNMATCHED ===")
for c in unmatched:
    print(c)

category_to_chunks = {}

for r in matched:
    mcId = r["mcId"]

    if mcId not in category_to_chunks:
        category_to_chunks[mcId] = []

    category_to_chunks[mcId].append(r["chunk"])

print('\n-------------------------------------------------------------\n')
print(category_to_chunks)

In [ ]:
# ---------- 1. Загрузка маппинга id -> название (из JSON) ----------
def load_mc_mapping2(json_path='dic_of_mc_key_phrases.json'):
    """Возвращает словарь {mcId: mcTitle} и список всех id"""
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    id_to_title = {item['mcId']: item['mcTitle'] for item in data}
    all_mc_ids = sorted(id_to_title.keys())
    return id_to_title, all_mc_ids

id_to_title, ALL_MC_IDS = load_mc_mapping2()
IDX_TO_ID = {i: mc_id for i, mc_id in enumerate(ALL_MC_IDS)}
ID_TO_IDX = {mc_id: i for i, mc_id in enumerate(ALL_MC_IDS)}

In [ ]:
# ---------- 2. Загрузка моделей и токенизатора ----------
tokenizer = AutoTokenizer.from_pretrained("deepvk/USER2-base")

detected_model = AutoModelForSequenceClassification.from_pretrained("./final_detected_model")
detected_model.eval()

split_model = AutoModelForSequenceClassification.from_pretrained("./final_split_model")
split_model.eval()

In [ ]:
# ---------- 3. Функция подготовки текста для модели ----------
def prepare_text(mc_title, mc_id, cleaned_description):
    """Если mc_id == 'unknown', не добавляем часть про ID."""
    if mc_id == "unknown":
        return f"Исходная категория: {mc_title}. {cleaned_description}"
    else:
        return f"Исходная категория: {mc_title} (ID {mc_id}). {cleaned_description}"

In [ ]:
# ---------- 4. Функция предсказания (без device) ----------
def predict_mc_ids(model, text, threshold=0.5):
    """Возвращает список mcId, для которых предсказанная вероятность > threshold."""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512, padding="max_length")
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.sigmoid(logits).cpu().numpy()[0]
    predicted_indices = np.where(probs > threshold)[0]
    return [IDX_TO_ID[i] for i in predicted_indices]

In [ ]:
# ---------- 5. Основная функция обработки одного объявления ----------
def process_single_ad(classified, cleaned_description, category_to_chunks):
    """
    classified: dict с ключами 'itemId', 'mcId', 'mcTitle', 'description'
    cleaned_description: str, очищенный текст
    category_to_chunks: dict {mcId: [chunk1, chunk2, ...]}
    """
    mc_id = classified.get("mcId", "unknown")
    mc_title = classified.get("mcTitle", "Неизвестная категория")
    
    # Формируем текст для модели
    model_input_text = prepare_text(mc_title, mc_id, cleaned_description)
    
    # 1. Detected модель – определяем все найденные услуги
    detected_mc_ids = predict_mc_ids(detected_model, model_input_text, threshold=0.5)
    
    # 2. Split модель – определяем, какие услуги надо вынести в отдельные черновики
    split_mc_ids = predict_mc_ids(split_model, model_input_text, threshold=0.5)
    should_split = len(split_mc_ids) > 0
    
    # 3. Формируем черновики для каждой категории из split_mc_ids
    drafts = []
    for mc_id_split in split_mc_ids:
        # Название категории
        title = id_to_title.get(mc_id_split, "Неизвестная категория")
        # Берём чанки из category_to_chunks, если есть
        chunks = category_to_chunks.get(mc_id_split, [])
        if chunks:
            # Склеиваем через точку с запятой (можно с пробелом после)
            text_draft = "; ".join(chunks)
        else:
            text_draft = cleaned_description
        
        drafts.append({
            "mcId": mc_id_split,
            "mcTitle": title,
            "text": text_draft
        })
    
    # 4. Итоговый результат
    result = {
        "detectedMcIds": detected_mc_ids,
        "shouldSplit": should_split,
        "drafts": drafts
    }
    return result


In [ ]:
# Вызов функции обработки
draft_creator = process_single_ad(classified, cleaned_description, category_to_chunks)

# Вывод результата в красивом JSON-формате (для удобства)
print(json.dumps(draft_creator, ensure_ascii=False, indent=2))

In [ ]:
# =============================================
# === ГЕНЕРАЦИЯ ЧИСТОГО ТЕКСТА ДЛЯ ЧЕРНОВИКОВ ===
# =============================================

# Загружаем модель один раз (в начале скрипта, после парсинга)
DRAFT_GEN_MODEL_NAME = "RussianNLP/FRED-T5-Summarizer"

draft_gen_tokenizer = AutoTokenizer.from_pretrained(DRAFT_GEN_MODEL_NAME)
draft_gen_model = AutoModelForSeq2SeqLM.from_pretrained(
    DRAFT_GEN_MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"          # автоматически выберет GPU или CPU — принудительность убрана
)
draft_gen_model.eval()

def generate_draft_description_text(mc_title: str, raw_draft_text: str) -> str:
    """
    Генерирует чистое описание для черновика.
    Работает и на GPU, и на CPU автоматически.
    """
    
    # Формируем вход для модели
    draft_gen_input_text = (
        f"<LM> Составь описание услуги. "
        f"Категория: {mc_title}. "
        f"Текст: {raw_draft_text}. "
        f"Начинай СТРОГО со слов 'Услуги ' и заканчивай 'могут быть сделаны отдельно.' "
        f"Перечисли только релевантные услуги, убери весь мусор и повторы."
    )
    
    # Токенизация
    inputs = draft_gen_tokenizer(
        draft_gen_input_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    
    # ← ИСПРАВЛЕНИЕ: переносим все тензоры на то же устройство, где стоит модель
    # (GPU или CPU — автоматически)
    device = next(draft_gen_model.parameters()).device
    inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else v 
              for k, v in inputs.items()}
    
    # Генерация
    draft_outputs = draft_gen_model.generate(
        **inputs,
        max_new_tokens=180,
        num_beams=5,
        early_stopping=True,
        no_repeat_ngram_size=2,
        temperature=0.65,
        do_sample=False,
        repetition_penalty=1.2
    )
    
    result = draft_gen_tokenizer.decode(draft_outputs[0], skip_special_tokens=True).strip()
    
    # Страховка формата
    if not result.startswith("Услуги "):
        result = f"Услуги {result}"
    if not result.endswith("могут быть сделаны отдельно."):
        result = result.rstrip(".") + " могут быть сделаны отдельно."
    
    return result

In [ ]:
# =============================================
# === ФИНАЛЬНАЯ ЯЧЕЙКА: обработка + генерация чистых описаний ===
# =============================================

# 2. Если есть сплит (то есть есть черновики), заменяем сырой текст на чистый
if draft_creator.get("shouldSplit", False) and draft_creator.get("drafts"):
    print("🔄 Генерируем чистые описания для черновиков...")
    
    for draft in draft_creator["drafts"]:
        mc_title = draft["mcTitle"]
        raw_text = draft["text"]                    # это либо чанки, либо cleaned_description
        
        # Генерируем красивое описание
        clean_text = generate_draft_description_text(mc_title, raw_text)
        
        # Заменяем текст в черновике
        draft["text"] = clean_text
        print(f"   ✓ {mc_title} — текст сгенерирован")
    
    print("✅ Все описания черновиков очищены и готовы")
else:
    print("ℹ️  shouldSplit = False → черновики не нужны, генерация пропущена")

# 3. Красивый вывод финального результата
print("\n" + "="*60)
print("ФИНАЛЬНЫЙ РЕЗУЛЬТАТ ДЛЯ ОБЪЯВЛЕНИЯ")
print("="*60)
print(json.dumps(draft_creator, ensure_ascii=False, indent=2))